# Pendule inversé : LQR et PPO — démonstration

Ce notebook **utilise** le package `pendule` (dossier `src/`) ; il n'entraîne aucun modèle.
Les agents PPO s'entraînent avec `scripts/entrainer_ppo.py`, et les résultats complets sont
produits par `scripts/evaluer.py`, `scripts/comparer.py` et `scripts/bassin.py`.

1. Paramètres (`config.yaml`)
2. Modèle linéarisé, stabilité, contrôlabilité
3. LQR de référence
4. Limite physique imposée par la saturation à 4 N
5. LQR sur le modèle non linéaire, avec et sans saturation
6. Résultats PPO / LQR (si `resultats/` existe)

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if Path.cwd().name == "notebooks":  # chemins relatifs à la racine du dépôt
    os.chdir("..")

from pendule.config import charger
from pendule.dynamique import angle_limite_rattrapage, linearisation
from pendule.lqr import ControleurLQR, gain_lqr
from pendule.metriques import metriques
from pendule.simulation import etat_initial, simuler

cfg = charger()
cfg.physique, cfg.simulation

## 2. Modèle linéarisé

$s = [x, \dot x, \theta, \dot\theta]$, $\theta = 0$ en haut. Autour de $s = 0$ : $\dot s = A s + B u$.

In [ ]:
A, B = linearisation(cfg.physique)
print("A =\n", A.round(3))
print("B =", B.ravel())
print("valeurs propres de A :", np.sort(np.linalg.eigvals(A).real).round(3))
kalman = np.hstack([np.linalg.matrix_power(A, k) @ B for k in range(4)])
print("rang de la matrice de Kalman :", np.linalg.matrix_rank(kalman))

## 3. LQR de référence : $Q = \mathrm{diag}(1, 0, 10, 0)$, $R = 0{,}1$

In [ ]:
K = gain_lqr(cfg)
print("K =", K.round(2))
print("valeurs propres de A − BK :", np.linalg.eigvals(A - B @ K[None, :]).round(2))

## 4. Limite physique

Avec $|u| \le u_{max}$ et $\dot\theta_0 = 0$, le pendule ne peut plus être redressé si
$\tan\theta_0 > u_{max} / ((M+m)g)$ : $\ddot\theta > 0$ quelle que soit la commande.

In [ ]:
print(f"θ* = {np.rad2deg(angle_limite_rattrapage(cfg.physique, cfg.simulation.u_max)):.2f}°")

## 5. LQR sur le modèle non linéaire (protocole d'évaluation commun)

In [ ]:
lignes = []
fig, ax = plt.subplots(figsize=(8, 3.5))
for u_max, style in [(cfg.simulation.u_max, "-"), (float("inf"), ":")]:
    c = cfg.modifier("simulation", u_max=u_max)
    for angle in [5.0, 15.0, 18.0, 20.0]:
        tr = simuler(ControleurLQR(c), etat_initial(angle), c)
        ax.plot(tr.t, np.rad2deg(tr.etats[:, 2]), style, label=f"{angle:g}°, u_max = {u_max:g} N")
        m = metriques(tr, c)
        lignes.append(
            {
                "u_max_N": u_max,
                "theta0_deg": angle,
                **{k: m[k] for k in ["issue", "t_etablissement_s", "u_demande_max_N"]},
            }
        )
ax.set(xlabel="temps (s)", ylabel="θ (°)", title="LQR saturé (traits pleins) et non saturé (pointillés)")
ax.legend(fontsize=7, ncol=2)
plt.show()
pd.DataFrame(lignes)

## 6. Résultats complets (produits par les scripts)

In [ ]:
resume = Path("resultats/resume.csv")
if resume.exists():
    df = pd.read_csv(resume)
    display(
        df[
            [
                "controleur",
                "theta0_deg",
                "n_reussites",
                "n_essais",
                "t_etablissement_s_moy",
                "t_etablissement_s_std",
                "issues",
            ]
        ]
    )
    print(Path("resultats/diapos_vs_recalcule.md").read_text())
else:
    print("Lancer d'abord scripts/evaluer.py et scripts/comparer.py (voir README).")

In [ ]:
from IPython.display import Image, display

for nom in ["reussite.png", "comparaison_trajectoires.png", "bassin_attraction.png", "euler_vs_rk4.png"]:
    if Path("figures", nom).exists():
        display(Image(filename=f"figures/{nom}"))